# Anima LoRA CLI · Colab T4

GPU 런타임 → 설치 → 데이터 → TOML → 모델·전처리 → smoke(선택) → TensorBoard → 본 학습.

- 기본값: **2,400스텝 · batch 4 · 512 해상도**, 정기 저장 200스텝.
- 정기 저장은 **런타임 내부**입니다. 다음 세션에 이어 하려면 수동 정지 시 **GD 보관**을 체크하거나 백업을 다운로드하세요.
- 설치는 [Anima-lora](https://github.com/sorryhyun/anima_lora)의 지정 커밋을 `git clone`합니다.


In [ ]:
#@title 1. 작업 경로와 설치
REPO_REVISION = "2b5d618619e3c58f503425b6fa6e47487f35a687"  #@param {type:"string"}
WORK_ROOT = "/content/anima_cli"  #@param {type:"string"}
MOUNT_DRIVE = False  #@param {type:"boolean"}

import os, sys, json, subprocess, shutil, tarfile, zipfile, stat, time, re, math
import urllib.request
import urllib.parse
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from copy import deepcopy

if not Path("/content").is_dir():
    raise RuntimeError("Google Colab의 GPU 런타임에서 실행하세요.")

if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

ROOT = Path(WORK_ROOT).expanduser().resolve()
if not ROOT.is_relative_to(Path("/content")) or ROOT.is_relative_to(
    Path("/content/drive")
):
    raise ValueError("WORK_ROOT는 GD가 아닌 /content 내부 경로로 지정하세요.")
ROOT.mkdir(parents=True, exist_ok=True)
TOOLS = ROOT / "tools"
TOOLS.mkdir(exist_ok=True)
ENV = os.environ.copy()
ENV["PYTHONUNBUFFERED"] = "1"
ENV["HF_HUB_DISABLE_TELEMETRY"] = "1"
PROCESSES = globals().get("PROCESSES", {})


def run(cmd, *, cwd=None, capture=False, log=None):
    cmd = [str(x) for x in cmd]
    if capture:
        result = subprocess.run(cmd, cwd=cwd, env=ENV, text=True, capture_output=True)
        if result.returncode:
            print(result.stdout[-6000:])
            print(result.stderr[-6000:])
            raise RuntimeError(f"명령 실패({result.returncode}): {cmd[0]}")
        return result.stdout
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    handle = Path(log).open("w", encoding="utf-8") if log else None
    try:
        for line in proc.stdout:
            print(line, end="")
            if handle:
                handle.write(line)
                handle.flush()
        if proc.wait():
            raise RuntimeError(f"명령 실패({proc.returncode}). 위 로그를 확인하세요.")
    except BaseException:
        import signal

        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGTERM)
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait()
        raise
    finally:
        if handle:
            handle.close()


def fetch_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "anima-colab"})
    with urllib.request.urlopen(req, timeout=60) as response:
        return json.load(response)


def safe_unzip(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            name = info.filename.replace("\\", "/")
            target = (destination / name).resolve()
            if (
                PurePosixPath(name).is_absolute()
                or not target.is_relative_to(destination)
                or stat.S_ISLNK(info.external_attr >> 16)
            ):
                raise ValueError(f"허용되지 않는 ZIP 항목: {name}")
        z.extractall(destination)


if not re.fullmatch(r"[0-9a-fA-F]{40}", REPO_REVISION):
    raise ValueError(
        "재현 가능한 설치를 위해 REPO_REVISION에 40자리 Git commit SHA를 넣으세요."
    )

REPO_URL = "https://github.com/sorryhyun/anima_lora.git"
REPO = ROOT / ("anima_lora_" + REPO_REVISION[:12])

if REPO.exists() and not (REPO / ".git").exists():
    REPO = ROOT / ("anima_lora_" + REPO_REVISION[:12] + "_git")

if REPO.exists() and not (REPO / ".git").exists():
    raise RuntimeError(f"Git 설치 경로에 다른 폴더가 있습니다: {REPO}")

if not REPO.exists():
    from tempfile import TemporaryDirectory

    # clone/fetch 실패 시 불완전한 설치 폴더가 남지 않도록 임시 위치에서 준비합니다.
    with TemporaryDirectory(prefix="anima_clone_", dir=ROOT) as temporary:
        checkout = Path(temporary) / "repo"
        run(
            [
                "git",
                "clone",
                "--no-checkout",
                "--depth",
                "1",
                REPO_URL,
                checkout,
            ]
        )
        run(
            ["git", "fetch", "--depth", "1", "origin", REPO_REVISION],
            cwd=checkout,
        )
        run(["git", "checkout", "--detach", REPO_REVISION], cwd=checkout)
        checkout.rename(REPO)

installed_revision = run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    capture=True,
).strip()

if installed_revision.lower() != REPO_REVISION.lower():
    raise RuntimeError(
        "설치된 Git 커밋이 REPO_REVISION과 다릅니다. "
        f"현재: {installed_revision}, 요청: {REPO_REVISION}. "
        "기존 체크아웃은 변경하지 않았습니다. 다른 WORK_ROOT를 사용하세요."
    )

print("Git 저장소 준비 완료:", REPO)
print("고정 커밋:", installed_revision)

ENV["ANIMA_HOME"] = str(REPO)
ENV["ANIME_TOOLS_HOME"] = str(REPO)
UV = TOOLS / "uv"

if not UV.exists():
    release = fetch_json("https://api.github.com/repos/astral-sh/uv/releases/latest")
    asset = next(
        x
        for x in release["assets"]
        if x["name"] == "uv-x86_64-unknown-linux-gnu.tar.gz"
    )
    archive = TOOLS / "uv.tar.gz"
    urllib.request.urlretrieve(asset["browser_download_url"], archive)
    with tarfile.open(archive) as tar:
        member = next(
            x
            for x in tar.getmembers()
            if x.isfile() and PurePosixPath(x.name).name == "uv"
        )
        with tar.extractfile(member) as src, UV.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    UV.chmod(0o755)

# lock에 고정된 원격 anime-tools만 설치합니다.
# --locked는 제외된 개발 그룹의 ../anime_tools 메타데이터까지 확인합니다.
run(
    [
        UV,
        "sync",
        "--frozen",
        "--python",
        "3.13",
        "--no-default-groups",
        "--group",
        "anime-tools-git",
    ],
    cwd=REPO,
)
PY = REPO / ".venv/bin/python"


def py(source, *args, capture=False):
    return run([PY, "-c", source, *args], cwd=REPO, capture=capture)


def write_toml(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    py(
        'import json,sys,toml; from pathlib import Path; Path(sys.argv[1]).write_text(toml.dumps(json.loads(sys.argv[2])), encoding="utf-8")',
        str(path),
        json.dumps(data, ensure_ascii=False),
    )


def read_toml(path):
    return json.loads(
        py(
            'import json,sys,tomllib; print(json.dumps(tomllib.load(open(sys.argv[1],"rb"))))',
            str(path),
            capture=True,
        )
    )


run(["nvidia-smi"])

# 설치를 다시 실행해도 T4 호환성 처리를 자동 적용합니다.
gpu = json.loads(
    py(
        "import json,torch,importlib.util; print(json.dumps({'major': torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else None, 'flash_installed': importlib.util.find_spec('flash_attn') is not None}))",
        capture=True,
    )
)

if gpu["major"] is not None and gpu["major"] < 8:
    if gpu["flash_installed"]:
        run([UV, "pip", "uninstall", "--python", PY, "flash-attn"], cwd=REPO)
    py(
        "from networks import attention_dispatch as a; assert a.flash_attn_varlen_func is None; assert a.flash_attn_func is None; print('T4: 텍스트 캐시용 SDPA 전환 확인 완료')"
    )
else:
    print("FlashAttention 자동 제거 대상 아님. GPU capability major:", gpu["major"])

SMOKE_OK = None
RESTORED = None
RESTORED_STATE = None
RESTORED_MANIFEST = None
(ROOT / "environment.txt").write_text(
    run([UV, "pip", "freeze", "--python", PY], capture=True), encoding="utf-8"
)
print("설치 완료:", REPO, "\n학습 Python:", PY)

In [ ]:
#@title 학습 재개·진단 도우미 준비 (설치 후 실행)
# 도우미가 노트북에 내장되어 있어 .ipynb 하나만으로 실행할 수 있습니다.
RUNTIME_SOURCE = r'''
"""코랩 노트북에 내장되는 상태 보관·복원 및 짧은 학습 진단 도우미."""

import argparse
import json
import math
import os
from pathlib import Path, PurePosixPath
import runpy
import shutil
import stat
import sys
import time
import zipfile


class SavedTrainingStop(KeyboardInterrupt):
    """업데이트 경계에서 저장을 마친 수동 정지입니다."""


def write_control(path, data):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".partial")
    temporary.write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporary.replace(path)


def copy_manual_snapshot(snapshot, destination):
    """완료된 스냅샷을 새 폴더로 복사하고, 완료 후에만 최종 이름을 붙입니다."""
    snapshot, destination = Path(snapshot), Path(destination)
    inspect_state(snapshot / "state")
    if not (snapshot / "complete.json").is_file():
        raise ValueError("수동 저장이 완료되지 않았습니다.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        raise FileExistsError(destination)
    partial = destination.with_name(destination.name + ".partial")
    shutil.copytree(snapshot, partial)
    # state_folder 복원에서 GD에 복사된 로그를 찾도록 경로를 갱신합니다.
    metadata_path = partial / "state" / "colab_state.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    metadata["config"]["logging_dir"] = str(destination / "logs")
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    partial.rename(destination)


def save_manual_stop(state, request, config, config_path, writer=None):
    """완료된 optimizer step의 LoRA와 전체 재개 상태를 함께 저장합니다."""
    if not state.accelerator.is_main_process:
        raise RuntimeError("코랩 수동 정지는 단일 프로세스 학습용입니다.")
    name = f"manual-{state.global_step:07d}-{time.time_ns()}"
    snapshot = Path(config["output_dir"]) / name
    snapshot.mkdir(parents=True, exist_ok=False)
    state.optimizer_eval_fn()
    # 저장소 pre-hook은 current_step + 1을 기록합니다.
    state.current_step.value = state.global_step - 1
    state.accelerator.save_state(str(snapshot / "state"))
    state.saver.save(
        str(Path(name) / "lora.safetensors"),
        state.network,
        state.global_step,
        state.current_epoch.value,
    )
    state.accelerator.end_training()
    if writer:
        writer.flush()
    shutil.copy2(config_path, snapshot / "training.toml")
    logs = Path(config["logging_dir"])
    if logs.is_dir():
        shutil.copytree(logs, snapshot / "logs")
    result = dict(
        status="saved",
        step=state.global_step,
        snapshot=str(snapshot),
        state=str(snapshot / "state"),
    )
    write_control(snapshot / "complete.json", result)
    if request.get("drive_folder"):
        destination = Path(request["drive_folder"]) / name
        try:
            copy_manual_snapshot(snapshot, destination)
            result["drive_snapshot"] = str(destination)
        except Exception as error:
            result["drive_error"] = str(error)
    return result


def install_stop_hook(loop, control_path, config, config_path, get_writer):
    """스텝 로그 기록 뒤, 누적 gradient 업데이트가 완료된 때만 정지합니다."""
    original = loop._log_step

    def log_and_stop(trainer, state, *args, **kwargs):
        original(trainer, state, *args, **kwargs)
        if not state.accelerator.sync_gradients or not Path(control_path).is_file():
            return
        request = json.loads(Path(control_path).read_text(encoding="utf-8"))
        if request.get("status") != "requested":
            return
        write_control(control_path, dict(request, status="saving"))
        try:
            result = save_manual_stop(state, request, config, config_path, get_writer())
        except Exception as error:
            write_control(control_path, dict(status="save_failed", error=str(error)))
            raise
        write_control(control_path, result)
        print("수동 저장 완료:", result, flush=True)
        raise SavedTrainingStop()

    loop._log_step = log_and_stop
    return original


def start_training_controls(command, cwd, env, log, control, save_to_drive=False):
    """학습 프로세스와 별도로 동작하는 저장 후 정지 버튼을 표시합니다."""
    import subprocess
    import threading
    import ipywidgets as widgets
    from IPython.display import display

    drive_root = Path("/content/drive/MyDrive")
    if save_to_drive and not drive_root.is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    checkbox = widgets.Checkbox(
        value=save_to_drive, description="정지 시 GD에도 보관", indent=False
    )
    destination = widgets.Text(
        value=str(drive_root / "anima_cli/manual_saves"),
        description="GD 폴더:",
        layout=widgets.Layout(width="95%"),
    )
    button = widgets.Button(
        description="현재 상태 저장 후 정지",
        button_style="warning",
        layout=widgets.Layout(width="220px"),
    )
    status = widgets.Label(value="학습 중 — 정기 저장은 런타임 내부에만 저장됩니다.")
    output = widgets.Output(layout={"height": "320px", "overflow": "auto"})
    Path(control).unlink(missing_ok=True)
    with Path(log).open("w", encoding="utf-8") as handle:
        process = subprocess.Popen(
            [str(v) for v in command],
            cwd=cwd,
            env=env,
            stdout=handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    def request_stop(_):
        if process.poll() is not None:
            status.value = "이미 종료되었습니다. 아래 로그를 확인하세요."
            return
        folder = None
        if checkbox.value:
            folder = Path(destination.value).expanduser().resolve()
            if not drive_root.is_dir() or not folder.is_relative_to(
                drive_root.resolve()
            ):
                status.value = (
                    "GD를 먼저 마운트하고 MyDrive 안의 보관 폴더를 지정하세요."
                )
                return
        write_control(
            control,
            dict(status="requested", drive_folder=str(folder) if folder else None),
        )
        button.disabled = checkbox.disabled = destination.disabled = True
        status.value = "정지 요청됨 — 현재 업데이트를 마친 뒤 저장합니다. 저장 완료까지 기다려 주세요."

    button.on_click(request_stop)
    display(widgets.VBox([checkbox, destination, button, status, output]))

    def monitor():
        try:
            with Path(log).open(encoding="utf-8", errors="replace") as handle:
                while True:
                    chunk = handle.read()
                    if chunk:
                        output.append_stdout(chunk)
                    if process.poll() is not None:
                        output.append_stdout(handle.read())
                        break
                    time.sleep(1)
            result = (
                json.loads(Path(control).read_text(encoding="utf-8"))
                if Path(control).exists()
                else {}
            )
            if result.get("status") == "saved":
                status.value = f"저장 후 정지 완료: {result['step']} steps"
                output.append_stdout(
                    "\n재개 상태: "
                    + result["state"]
                    + "\nLoRA: "
                    + str(Path(result["snapshot"]) / "lora.safetensors")
                    + "\n"
                )
                if result.get("drive_snapshot"):
                    output.append_stdout(
                        "GD 보관 완료: " + result["drive_snapshot"] + "\n"
                    )
                if result.get("drive_error"):
                    status.value = "런타임 저장 완료 / GD 복사 실패 — 로컬 상태를 보관한 뒤 종료하세요."
                    output.append_stdout(result["drive_error"] + "\n")
            else:
                status.value = (
                    "학습 완료"
                    if process.returncode == 0
                    else f"학습 실패 ({process.returncode}) — 로그를 확인하세요."
                )
        finally:
            button.disabled = checkbox.disabled = destination.disabled = True

    threading.Thread(target=monitor, daemon=True).start()
    return process


def inspect_state(folder):
    folder = Path(folder)
    info = json.loads((folder / "train_state.json").read_text(encoding="utf-8"))
    step = info.get("current_step")
    if not isinstance(step, int) or isinstance(step, bool) or step < 0:
        raise ValueError("학습 상태의 current_step이 올바르지 않습니다.")
    names = [p.name for p in folder.iterdir() if p.is_file()]
    required = {
        "모델": any(
            n.startswith(("model", "pytorch_model"))
            and n.endswith((".safetensors", ".bin"))
            for n in names
        ),
        "optimizer": any(
            n.startswith("optimizer") and n.endswith(".bin") for n in names
        ),
        "scheduler": any(
            n.startswith("scheduler") and n.endswith(".bin") for n in names
        ),
        "난수 상태": any(
            n.startswith("random_states") and n.endswith(".pkl") for n in names
        ),
    }
    missing = [key for key, present in required.items() if not present]
    if missing:
        raise ValueError("불완전한 상태 폴더: " + ", ".join(missing))
    return info


def list_states(folder, complete_only=False):
    result = []
    for path in Path(folder).rglob("train_state.json"):
        try:
            if complete_only and not (path.parent / "colab_state.json").is_file():
                continue
            info = inspect_state(path.parent)
            result.append((info["current_step"], path.parent))
        except (OSError, ValueError):
            continue
    return sorted(result, key=lambda item: (item[0], str(item[1])))


def create_bundle(
    destination, state, config, config_path, revision, root, repo, preparation=None
):
    """종료된 학습의 완성된 상태 하나와 로그를 새 ZIP으로 보관합니다."""
    destination, state = Path(destination), Path(state)
    info = inspect_state(state)
    manifest = dict(
        format_version=1,
        revision=revision,
        root=str(root),
        repo=str(repo),
        saved_step=info["current_step"],
        config=config,
        preparation=preparation,
    )
    # 미완료 ZIP은 복원 가능한 백업처럼 보이지 않도록 .partial로 남깁니다.
    if destination.exists():
        raise FileExistsError(destination)
    partial = destination.with_suffix(destination.suffix + ".partial")
    with zipfile.ZipFile(partial, "x", compression=zipfile.ZIP_STORED) as archive:
        archive.writestr(
            "manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2)
        )
        archive.write(config_path, "training.toml")
        for base, prefix in ((state, "state"), (Path(config["logging_dir"]), "logs")):
            if base.exists():
                for path in sorted(base.rglob("*")):
                    if path.is_symlink():
                        raise ValueError("백업에 심볼릭 링크를 포함할 수 없습니다.")
                    if path.is_file():
                        archive.write(
                            path,
                            str(
                                PurePosixPath(prefix)
                                / path.relative_to(base).as_posix()
                            ),
                        )
    partial.rename(destination)
    return manifest


def restore_bundle(archive_path, destination, revision):
    """자신이 만든 백업만 복원합니다. 기존 파일은 덮어쓰지 않습니다."""
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive_path) as archive:
        manifest = json.loads(archive.read("manifest.json"))
        if manifest.get("format_version") != 1 or manifest.get("revision") != revision:
            raise ValueError(
                "백업 형식 또는 REPO_REVISION이 다릅니다. 저장 당시 revision으로 설치하세요."
            )
        seen = set()
        for item in archive.infolist():
            name = item.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            target = (destination / name).resolve()
            if (
                relative.is_absolute()
                or ".." in relative.parts
                or ":" in name
                or not target.is_relative_to(destination)
                or stat.S_ISLNK(item.external_attr >> 16)
                or name in seen
            ):
                raise ValueError("안전하지 않거나 중복된 ZIP 경로: " + name)
            seen.add(name)
        if destination.exists():
            raise FileExistsError("새 복원 폴더를 지정하세요: " + str(destination))
        free = shutil.disk_usage(destination.parent).free
        if sum(item.file_size for item in archive.infolist()) > free:
            raise ValueError("백업을 풀 디스크 공간이 부족합니다.")
        destination.mkdir()
        archive.extractall(destination)
    info = inspect_state(destination / "state")
    if info["current_step"] != manifest["saved_step"]:
        raise ValueError("백업 step과 상태 폴더의 step이 다릅니다.")
    return manifest


def remap_config(manifest, root, repo, output):
    """새 런타임의 작업·저장 경로에 맞춰 설정을 복원합니다."""
    config = json.loads(json.dumps(manifest["config"]))

    def remap(value):
        if isinstance(value, dict):
            return {k: remap(v) for k, v in value.items()}
        if isinstance(value, list):
            return [remap(v) for v in value]
        if isinstance(value, str):
            for old, new in ((manifest["repo"], repo), (manifest["root"], root)):
                if value == old or value.startswith(old.rstrip("/") + "/"):
                    return str(new) + value[len(old) :]
        return value

    config = remap(config)
    for key in ("resume", "initial_step", "initial_epoch", "log_tracker_config"):
        config.pop(key, None)
    config.update(
        output_dir=str(Path(output) / "ckpt"), logging_dir=str(Path(output) / "logs")
    )
    # 새 런타임에서 재다운로드한 모델의 mtime 차이로 이전 캐시를 섞지 않습니다.
    prepared = Path(root) / "prepared" / Path(output).name
    config.update(
        resized_image_dir=str(prepared / "resized"),
        lora_cache_dir=str(prepared / "cache"),
    )
    for dataset in config.get("datasets", []):
        for subset in dataset.get("subsets", []):
            subset.update(
                image_dir=config["resized_image_dir"],
                cache_dir=config["lora_cache_dir"],
            )
    return config


def summarize_probe(path):
    rows = [
        json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()
    ]
    result = {}
    for phase in ("baseline", "diagnostic"):
        selected = [row for row in rows if row["phase"] == phase]
        if selected:
            result[phase] = dict(
                micro_steps=len(selected),
                seconds_mean=sum(row["seconds"] for row in selected) / len(selected),
                peak_allocated_gib=max(row["peak_allocated"] for row in selected)
                / 2**30,
                peak_reserved_gib=max(row["peak_reserved"] for row in selected) / 2**30,
            )
    result["metrics"] = sorted({key for row in rows for key in row.get("metrics", {})})
    return result


def launch_training(
    config_path,
    probe_path=None,
    diagnostics=False,
    timeline=False,
    module_limit=3,
    session_path=None,
    control_path=None,
):
    """고정된 소스의 학습 루프를 프로세스 안에서만 감쌉니다."""
    import torch
    import tomllib
    from library.training import loop
    from library.training.repa import REPAMethodAdapter
    from torch.utils.tensorboard import SummaryWriter

    config = tomllib.loads(Path(config_path).read_text())
    if session_path:
        from accelerate import Accelerator

        session = json.loads(Path(session_path).read_text())
        original_save = Accelerator.save_state

        def save_complete(self, output_dir=None, *args, **kwargs):
            if output_dir is not None and self.is_main_process:
                (Path(output_dir) / "colab_state.json").unlink(missing_ok=True)
            result = original_save(self, output_dir, *args, **kwargs)
            folder = Path(result if result is not None else output_dir)
            if self.is_main_process:
                if writer:
                    writer.flush()
                metadata = dict(session, config=config)
                inspect_state(folder)
                temporary = folder / "colab_state.json.partial"
                temporary.write_text(
                    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
                )
                temporary.replace(folder / "colab_state.json")
            return result

        Accelerator.save_state = save_complete
    writer = None
    resume_step = (
        inspect_state(config["resume"])["current_step"]
        if config.get("resume")
        else None
    )
    if timeline:
        writer = SummaryWriter(
            str(Path(config["logging_dir"]) / "colab_timeline"),
            purge_step=resume_step + 1 if resume_step is not None else None,
            flush_secs=10,
        )
        # 저장소의 세션별 이벤트는 그대로 남기고, 같은 global_step으로 연속 기록합니다.
        from library.training import log_dispatch

        original_dispatch = log_dispatch.dispatch_logs

        def dispatch(
            accelerator, logs, step_value, global_step, epoch, val_step=None, **kwargs
        ):
            original_dispatch(
                accelerator, logs, step_value, global_step, epoch, val_step, **kwargs
            )
            if accelerator.is_main_process:
                for key, value in logs.items():
                    if isinstance(value, (int, float)) and math.isfinite(value):
                        writer.add_scalar(key, value, global_step)

        log_dispatch.dispatch_logs = dispatch

    original_step = loop._run_step
    original_repa_setup = REPAMethodAdapter.on_network_built
    if probe_path:
        Path(probe_path).parent.mkdir(parents=True, exist_ok=True)

        # smoke의 기본 구간에서는 기존 TOML의 heatmap 설정도 끕니다.
        def setup_repa(self, ctx):
            original_repa_setup(self, ctx)
            self._grad_heatmap_every = 0

        REPAMethodAdapter.on_network_built = setup_repa

    selected = None
    previous_grad = {}

    def measured_step(trainer, state, batch):
        nonlocal selected
        active = diagnostics and state.global_step >= 2
        network = state.accelerator.unwrap_model(state.network)
        if selected is None:
            candidates = [
                (n, m)
                for n, m in network.named_modules()
                if hasattr(m, "lora_down")
                and hasattr(m, "lora_up")
                and hasattr(m.lora_down, "weight")
                and m.lora_down.weight.ndim == 2
                and hasattr(m.lora_up, "weight")
                and m.lora_up.weight.ndim == 2
            ]
            count = min(module_limit, len(candidates))
            selected = [
                candidates[round(i * (len(candidates) - 1) / max(1, count - 1))]
                for i in range(count)
            ]
        for adapter in trainer._adapters:
            if isinstance(adapter, REPAMethodAdapter):
                adapter._grad_heatmap_every = (
                    1 if active and adapter._mode == "relational" else 0
                )
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        metrics, handles, before = {}, [], {}
        try:
            if active:
                for name, module in selected:
                    for kind in ("lora_down", "lora_up"):
                        parameter = getattr(module, kind).weight
                        key = name + "/" + kind
                        before[key] = parameter.detach().float().clone()
                        if parameter.requires_grad:

                            def capture(gradient, key=key):
                                # hook 시점의 gradient는 FP16 GradScaler 배율을 제거합니다.
                                scale = (
                                    state.accelerator.scaler.get_scale()
                                    if state.accelerator.scaler
                                    else 1.0
                                )
                                flat = gradient.detach().float().flatten() / scale
                                norm = flat.norm()
                                metrics[key + "/grad_norm"] = norm.item()
                                prior = previous_grad.get(key)
                                if prior is not None and norm > 0 and prior.norm() > 0:
                                    metrics[key + "/consecutive_grad_cosine"] = (
                                        torch.nn.functional.cosine_similarity(
                                            flat, prior.to(flat.device), dim=0
                                        ).item()
                                    )
                                previous_grad[key] = flat.clone()

                            handles.append(parameter.register_hook(capture))
            loss = original_step(trainer, state, batch)
            if active:
                with torch.no_grad():
                    for name, module in selected:
                        for kind in ("lora_down", "lora_up"):
                            key = name + "/" + kind
                            weight = getattr(module, kind).weight.detach().float()
                            metrics[key + "/update_norm"] = (
                                (weight - before[key].to(weight.device)).norm().item()
                            )
                        if config.get("use_timestep_mask") and hasattr(
                            module, "_timestep_mask"
                        ):
                            mask = module._timestep_mask.detach()
                            metrics[name + "/active_rank_mean"] = (
                                (mask != 0).float().sum(dim=-1).mean().item()
                            )
                    for adapter in trainer._adapters:
                        if isinstance(adapter, REPAMethodAdapter):
                            metrics.update(adapter._metrics)
            torch.cuda.synchronize()
            row = dict(
                step=state.global_step + 1,
                phase="diagnostic" if active else "baseline",
                optimizer_step=bool(state.accelerator.sync_gradients),
                seconds=time.perf_counter() - started,
                peak_allocated=torch.cuda.max_memory_allocated(),
                peak_reserved=torch.cuda.max_memory_reserved(),
                metrics=metrics,
            )
            if not all(math.isfinite(v) for v in metrics.values()):
                raise RuntimeError("진단 지표에 NaN/Inf가 있습니다.")
            with Path(probe_path).open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
            return loss
        finally:
            for handle in handles:
                handle.remove()

    if probe_path:
        loop._run_step = measured_step
    # GPU 모듈 로딩보다 먼저 저장된 상태를 검증했습니다. 학습 소스 파일은 수정하지 않습니다.
    sys.argv = ["train.py", "--config_file", str(config_path)]
    original_log_step = None
    if control_path:
        original_log_step = install_stop_hook(
            loop, control_path, config, config_path, lambda: writer
        )
    try:
        runpy.run_path("train.py", run_name="__main__")
    except SavedTrainingStop:
        pass
    finally:
        if original_log_step is not None:
            loop._log_step = original_log_step
        if writer:
            writer.close()


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--probe")
    parser.add_argument("--diagnostics", action="store_true")
    parser.add_argument("--timeline", action="store_true")
    parser.add_argument("--module-limit", type=int, default=3)
    parser.add_argument("--session")
    parser.add_argument("--control")
    args = parser.parse_args()
    if args.module_limit < 1:
        parser.error("--module-limit은 1 이상이어야 합니다.")
    # 실행 파일은 tools/에 있으므로 학습 저장소를 명시적으로 모듈 경로에 넣습니다.
    sys.path.insert(0, os.getcwd())
    launch_training(
        args.config,
        args.probe,
        args.diagnostics,
        args.timeline,
        args.module_limit,
        args.session,
        args.control,
    )
'''
RUNTIME = TOOLS / "colab_runtime.py"
RUNTIME.write_text(RUNTIME_SOURCE.removeprefix("\n"), encoding="utf-8")
import importlib.util

runtime_spec = importlib.util.spec_from_file_location("colab_runtime", RUNTIME)
colab_runtime = importlib.util.module_from_spec(runtime_spec)
runtime_spec.loader.exec_module(colab_runtime)
print("상태 저장·복원 및 smoke 진단 도우미 준비 완료")

## 2. GPU 실행 확인

CUDA·정밀도·SDPA 역전파를 확인합니다. 실패하면 설치 로그를 확인하세요.


In [ ]:
#@title GPU·연산 진단
py(r"""
import torch, json
if not torch.cuda.is_available():
    raise RuntimeError('CUDA를 사용할 수 없습니다. GPU 런타임과 드라이버/휠 호환성을 확인하세요.')
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(), 'capability:', torch.cuda.get_device_capability())
print('BF16 native:', torch.cuda.is_bf16_supported(including_emulation=False))
for dtype in (torch.bfloat16, torch.float16):
    try:
        q = torch.randn(1, 4, 128, 64, device='cuda', dtype=dtype, requires_grad=True)
        result = torch.nn.functional.scaled_dot_product_attention(q, q, q)
        result.float().square().mean().backward()
        assert torch.isfinite(result).all() and torch.isfinite(q.grad).all()
        torch.cuda.synchronize()
        print(dtype, 'SDPA forward/backward PASS')
    except Exception as exc:
        print(dtype, 'FAIL:', type(exc).__name__, str(exc))
""")

In [ ]:
#@title 3. 데이터 ZIP 업로드 또는 기존 폴더
DATA_MODE = "upload_zip"  #@param ["upload_zip", "zip_path", "folder"]
DATA_PATH = ""  #@param {type:"string"}

data_started = time.perf_counter()

if DATA_MODE in ("zip_path", "folder") and not DATA_PATH.strip():
    raise ValueError("DATA_PATH에 ZIP 파일 또는 데이터 폴더 경로를 입력하세요.")

if DATA_MODE == "upload_zip":
    from google.colab import files

    print(
        "[1/3] 브라우저 → 코랩 파일 전송 중입니다. 아직 압축을 풀지 않습니다.",
        flush=True,
    )
    stage_started = time.perf_counter()
    uploads = files.upload()
    print(
        f"[1/3] 파일 전송 완료: {time.perf_counter() - stage_started:.1f}초",
        flush=True,
    )
    if len(uploads) != 1:
        raise ValueError("이미지·캡션 ZIP 하나를 선택하세요.")
    name, content = next(iter(uploads.items()))
    archive = ROOT / Path(name).name
    print(f"ZIP 저장 중: {len(content) / 1024**2:.1f} MiB → {archive}", flush=True)
    stage_started = time.perf_counter()
    archive.write_bytes(content)
    print(f"ZIP 저장 완료: {time.perf_counter() - stage_started:.1f}초", flush=True)
    del uploads, content
elif DATA_MODE == "zip_path":
    archive = Path(DATA_PATH).expanduser().resolve()
elif DATA_MODE != "folder":
    raise ValueError(DATA_MODE)

if DATA_MODE == "folder":
    SOURCE = Path(DATA_PATH).expanduser().resolve()
    if not SOURCE.is_dir():
        raise NotADirectoryError(SOURCE)
    print("데이터 폴더:", flush=True)
else:
    if not archive.is_file() or not zipfile.is_zipfile(archive):
        raise ValueError("ZIP 경로를 확인하세요.")
    from tempfile import mkdtemp

    dataset_root = ROOT / "datasets"
    dataset_root.mkdir(parents=True, exist_ok=True)
    SOURCE = Path(mkdtemp(prefix="dataset_", dir=dataset_root))
    print(f"[2/3] 압축 해제 중: {SOURCE}", flush=True)
    stage_started = time.perf_counter()
    safe_unzip(archive, SOURCE)
    print(
        f"[2/3] 압축 해제 완료: {time.perf_counter() - stage_started:.1f}초",
        flush=True,
    )

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def image_files(folder):
    return sorted(
        p
        for p in Path(folder).rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTS
        and "__MACOSX" not in p.parts
        and not p.name.startswith("._")
    )


print(f"[3/3] 이미지·캡션 파일명 검사 중: {SOURCE}", flush=True)
stage_started = time.perf_counter()
images = image_files(SOURCE)

if not images:
    raise ValueError("학습 이미지가 없습니다.")

missing = [
    str(p.relative_to(SOURCE)) for p in images if not p.with_suffix(".txt").is_file()
]

if missing:
    raise ValueError(f"동일 이름 .txt 캡션 누락 {len(missing)}개: {missing[:10]}")

stems = [p.stem.casefold() for p in images]

if len(stems) != len(set(stems)):
    raise ValueError("캐시 이름 충돌을 피하도록 이미지 stem을 고유하게 바꾸세요.")

print(
    f"[3/3] 파일명 검사 완료: {time.perf_counter() - stage_started:.1f}초", flush=True
)
print("원본:", SOURCE, "| 이미지·캡션 쌍:", len(images))
print(f"전체 소요 시간: {time.perf_counter() - data_started:.1f}초", flush=True)

In [ ]:
#@title 3-1. 이전 학습 복원 (처음 학습할 때는 none)
RESTORE_MODE = "none"  #@param ["none", "zip_path", "upload_zip", "state_folder"]
RESTORE_PATH = ""  #@param {type:"string"}
RESTORE_OUTPUT_ROOT = ""  #@param {type:"string"}
# ZIP은 이 노트북의 백업 셀에서 만든 본인 파일만 사용하세요.
# state_folder는 재개 상태 폴더입니다. 수동 GD 보관은 manual-.../state를 지정하세요.
RESTORED = RESTORED_STATE = RESTORED_MANIFEST = None

if RESTORE_MODE != "none":
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    if RESTORE_MODE == "upload_zip":
        from google.colab import files

        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("학습 백업 ZIP 하나를 선택하세요.")
        name, content = next(iter(uploaded.items()))
        restore_zip = ROOT / ("resume_" + stamp + ".zip")
        restore_zip.write_bytes(content)
        del uploaded, content
    elif RESTORE_MODE == "zip_path":
        restore_zip = Path(RESTORE_PATH).expanduser().resolve()
    if RESTORE_MODE == "state_folder":
        RESTORED_STATE = Path(RESTORE_PATH).expanduser().resolve()
        colab_runtime.inspect_state(RESTORED_STATE)
        metadata_path = RESTORED_STATE / "colab_state.json"
        if not metadata_path.is_file():
            raise ValueError(
                "설정 기록이 없습니다. 3-1은 none으로 두고 TOML을 load한 뒤 4-1에서 상태를 선택하세요."
            )
        RESTORED_MANIFEST = json.loads(metadata_path.read_text())
        if RESTORED_MANIFEST["revision"] != REPO_REVISION:
            raise ValueError("저장 당시 REPO_REVISION으로 다시 설치하세요.")
    else:
        restored_dir = ROOT / ("restored_" + stamp)
        RESTORED_MANIFEST = colab_runtime.restore_bundle(
            restore_zip, restored_dir, REPO_REVISION
        )
        RESTORED_STATE = restored_dir / "state"
    default_root = ROOT / "output"
    target_parent = (
        Path(RESTORE_OUTPUT_ROOT).expanduser() if RESTORE_OUTPUT_ROOT else default_root
    )
    if not target_parent.resolve().is_relative_to(
        Path("/content")
    ) or target_parent.resolve().is_relative_to(Path("/content/drive")):
        raise ValueError(
            "복원 출력은 /content 내부로 지정하세요. GD는 수동 보관에만 사용합니다."
        )
    target = target_parent / ("resumed_" + stamp)
    if target.exists():
        raise FileExistsError(target)
    target.mkdir(parents=True)
    RESTORED = colab_runtime.remap_config(RESTORED_MANIFEST, ROOT, REPO, target)
    old_logs = (
        Path(RESTORED_MANIFEST["config"]["logging_dir"])
        if RESTORE_MODE == "state_folder"
        else restored_dir / "logs"
    )
    if old_logs.is_dir():
        shutil.copytree(old_logs, RESTORED["logging_dir"])
    print("복원된 상태:", RESTORED_STATE)
    print("저장 step:", colab_runtime.inspect_state(RESTORED_STATE)["current_step"])
    print("새 저장 위치:", target)
    print(
        "다음 TOML 셀은 복원한 설정을 자동으로 사용합니다. 원본 데이터와 모델·캐시는 별도로 준비합니다."
    )

## 4. TOML 생성 / 불러오기

- `form`: 아래 폼으로 설정합니다.
- `upload`: TOML을 업로드합니다. 같은 폴더의 `base_config`도 함께 올릴 수 있습니다. 여러 파일이면 `EXISTING_TOML`에 주 파일명을 입력하세요.
- `load`: `EXISTING_TOML`에 파일 경로를 입력합니다. 상대 경로는 학습 저장소 기준입니다.

업로드·불러오기는 파일의 HP를 사용하며, **3-1에서 복원한 설정이 최우선**입니다. 데이터는 한 subset을 사용합니다.
4-1에서 저장·재개를 설정합니다. `TOTAL_TARGET_STEPS=0`은 설정된 목표를 유지합니다.

| 기법 | 켜기 | 끄기 |
|---|---|---|
| SVD 초기화 | `down_init = "weight_svd"` | `down_init = "kaiming"` |
| T-LoRA | `use_timestep_mask = true` | `false` |
| REPA | `use_repa = true` | `false` |

REPA는 PE-Spatial 캐시를 추가합니다. Unsloth offload와 block swap은 동시에 사용할 수 없습니다.


In [ ]:
#@title TOML 설정 — upload/load는 파일의 HP 사용
CONFIG_MODE = "form"  #@param ["form", "upload", "load"]
EXISTING_TOML = ""  #@param {type:"string"}
RUN_NAME = "anima_lora"  #@param {type:"string"}
OUTPUT_ROOT = ""  #@param {type:"string"}
RESOLUTION = 512  #@param [512, 768, 896, 1024] {type:"raw"}
BATCH_SIZE = 4  #@param {type:"integer"}
NUM_REPEATS = 2  #@param {type:"integer"}
GRAD_ACCUM = 1  #@param {type:"integer"}
RANK = 32  #@param {type:"integer"}
ALPHA = 128  #@param {type:"integer"}
OPTIMIZER = "AdamW"  #@param ["AdamW", "Prodigy"]
LEARNING_RATE = 0.00002  #@param {type:"number"}
LR_SCHEDULER = "cosine"  #@param ["cosine", "constant", "constant_with_warmup"]
WARMUP = 0.05  #@param {type:"number"}
TRAIN_LIMIT = "steps"  #@param ["steps", "epochs"]
MAX_STEPS = 2400  #@param {type:"integer"}
MAX_EPOCHS = 8  #@param {type:"integer"}
SAVE_EVERY_STEPS = 200  #@param {type:"integer"}
SAVE_EVERY_EPOCHS = 0  #@param {type:"integer"}
SAVE_STATE = True  #@param {type:"boolean"}
SEED = 42  #@param {type:"integer"}
MIXED_PRECISION = "bf16"  #@param ["bf16", "fp16", "no"]
BLOCKS_TO_SWAP = 8  #@param {type:"integer"}
GRADIENT_CHECKPOINTING = True  #@param {type:"boolean"}
UNSLOTH_OFFLOAD = False  #@param {type:"boolean"}
TORCH_COMPILE = False  #@param {type:"boolean"}
ATTENTION = "torch"  #@param ["torch", "flash"]
SVD_INIT = True  #@param {type:"boolean"}
T_LORA = True  #@param {type:"boolean"}
MIN_RANK = 1  #@param {type:"integer"}
ALPHA_RANK_SCALE = 1.0  #@param {type:"number"}
REPA = True  #@param {type:"boolean"}
REPA_WEIGHT = 0.05  #@param {type:"number"}
REPA_LAYER = 8  #@param {type:"integer"}
CAPTION_DROPOUT = 0.1  #@param {type:"number"}
SHUFFLE_VARIANTS = 4  #@param {type:"integer"}
CJK_VOCAB_PACK = False  #@param {type:"boolean"}

if not OUTPUT_ROOT:
    OUTPUT_ROOT = str(ROOT / "output")

if CONFIG_MODE not in ("form", "upload", "load"):
    raise ValueError("CONFIG_MODE는 form, upload, load 중 하나여야 합니다.")

if RESTORED is not None:
    print("복원한 학습 설정을 사용합니다. form/upload/load 선택은 적용하지 않습니다.")
elif CONFIG_MODE == "upload":
    from google.colab import files
    import tomllib

    print(
        "학습 TOML을 선택하세요. 같은 폴더의 base_config 파일도 함께 선택할 수 있습니다."
    )
    uploaded_tomls = files.upload()

    if not uploaded_tomls:
        raise ValueError("TOML 업로드가 취소됐습니다.")

    for filename, content in uploaded_tomls.items():
        if (
            Path(filename).name != filename
            or "/" in filename
            or "\\" in filename
            or Path(filename).suffix.lower() != ".toml"
        ):
            raise ValueError("파일명만 있는 .toml 파일을 업로드하세요: " + filename)
        tomllib.loads(content.decode("utf-8-sig"))

    if len(uploaded_tomls) == 1:
        main_toml = next(iter(uploaded_tomls))
    else:
        main_toml = Path(EXISTING_TOML).name
        if main_toml not in uploaded_tomls:
            raise ValueError(
                "여러 TOML을 올릴 때는 EXISTING_TOML에 주 설정 파일명을 먼저 입력하세요. "
                "업로드한 파일: " + ", ".join(uploaded_tomls)
            )

    upload_dir = (
        ROOT / "configs" / "uploads" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    upload_dir.mkdir(parents=True, exist_ok=False)

    for filename, content in uploaded_tomls.items():
        (upload_dir / filename).write_text(
            content.decode("utf-8-sig"), encoding="utf-8"
        )

    EXISTING_TOML = str(upload_dir / main_toml)
    del uploaded_tomls, content
    print("업로드한 학습 설정:", EXISTING_TOML)

if RESTORED is not None:
    cfg = deepcopy(RESTORED)
elif CONFIG_MODE == "form":
    if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
        raise ValueError("RUN_NAME: 영문·숫자·밑줄·하이픈을 사용하세요.")
    cfg = json.loads(
        py(
            'import json; from library.config.io import load_method_preset; print(json.dumps(load_method_preset("lora","default")))',
            capture=True,
        )
    )
    cfg.update(
        network_dim=RANK,
        network_alpha=ALPHA,
        optimizer_type=OPTIMIZER,
        learning_rate=LEARNING_RATE,
        lr_scheduler=LR_SCHEDULER,
        lr_warmup_steps=WARMUP,
        gradient_accumulation_steps=GRAD_ACCUM,
        seed=SEED,
        mixed_precision=MIXED_PRECISION,
        blocks_to_swap=BLOCKS_TO_SWAP,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        unsloth_offload_checkpointing=UNSLOTH_OFFLOAD,
        torch_compile=TORCH_COMPILE,
        attn_mode=ATTENTION,
        down_init="weight_svd" if SVD_INIT else "kaiming",
        use_timestep_mask=T_LORA,
        min_rank=MIN_RANK,
        alpha_rank_scale=ALPHA_RANK_SCALE,
        use_repa=REPA,
        repa_weight=REPA_WEIGHT,
        repa_layer=REPA_LAYER,
        caption_dropout_rate=CAPTION_DROPOUT,
        use_shuffled_caption_variants=SHUFFLE_VARIANTS > 1,
        caption_shuffle_variants=SHUFFLE_VARIANTS,
        target_res=[RESOLUTION],
        save_state=SAVE_STATE,
        save_state_on_train_end=SAVE_STATE,
        save_every_n_steps=SAVE_EVERY_STEPS,
        save_every_n_epochs=SAVE_EVERY_EPOCHS or None,
        output_name=RUN_NAME,
        output_dir=str(Path(OUTPUT_ROOT) / RUN_NAME / "ckpt"),
        logging_dir=str(Path(OUTPUT_ROOT) / RUN_NAME / "logs"),
        log_with="tensorboard",
        log_every_n_steps=1,
        save_model_as="safetensors",
    )
    # Epoch 제한은 max_train_steps를 덮어쓰므로 하나만 저장합니다.
    cfg.pop("max_train_epochs", None)
    cfg.pop("max_train_steps", None)
    cfg["max_train_steps" if TRAIN_LIMIT == "steps" else "max_train_epochs"] = (
        MAX_STEPS if TRAIN_LIMIT == "steps" else MAX_EPOCHS
    )
    cfg.pop("checkpointing_epochs", None)
    if not CJK_VOCAB_PACK:
        cfg["vocab_pack"] = ""
    data_root = ROOT / "prepared" / RUN_NAME
    cfg["resized_image_dir"] = str(data_root / "resized")
    cfg["lora_cache_dir"] = str(data_root / "cache")
    cfg["general"] = {}
    cfg["datasets"] = [
        dict(
            batch_size=BATCH_SIZE,
            validation_split_num=0,
            repeat_by_folder_name=False,
            subsets=[
                dict(
                    image_dir=cfg["resized_image_dir"],
                    cache_dir=cfg["lora_cache_dir"],
                    num_repeats=NUM_REPEATS,
                    recursive=True,
                )
            ],
        )
    ]
else:
    if not EXISTING_TOML.strip() or not Path(EXISTING_TOML).is_file():
        raise ValueError("EXISTING_TOML 경로를 확인하세요.")
    cfg = json.loads(
        py(
            r"""
import json, sys, toml
from pathlib import Path
from library.config.io import _load_toml_with_base, load_dataset_config_from_base
p = str(Path(sys.argv[1]).resolve())
def inherited_blueprint(path, seen=None):
    seen = set() if seen is None else seen
    path = Path(path).resolve()
    if path in seen: raise ValueError('base_config cycle')
    seen.add(path)
    raw = toml.load(path)
    base = raw.get('base_config')
    result = inherited_blueprint(path.parent / base, seen) if base else {}
    result.update({k: raw[k] for k in ('general', 'datasets') if k in raw})
    return result
blueprint = inherited_blueprint(p)
c = _load_toml_with_base(p)
if c.get('dataset_config'):
    c.update(toml.load(c['dataset_config']))
    c.pop('dataset_config')
else:
    c.update(blueprint if blueprint.get('datasets') else (load_dataset_config_from_base(config_file=p, overrides=c) or {}))
print(json.dumps(c))
""",
            EXISTING_TOML,
            capture=True,
        )
    )

# 실행 제어 키는 standalone TOML에 포함하지 않습니다.
for key in (
    "method",
    "preset",
    "methods_subdir",
    "config_file",
    "print_config",
    "output_config",
    "config_snapshot",
):
    cfg.pop(key, None)

if cfg.get("unsloth_offload_checkpointing") and cfg.get("blocks_to_swap", 0):
    raise ValueError("Unsloth offload와 block swap 중 하나만 선택하세요.")

if cfg.get("use_repa") and cfg.get("repa_encoder", "pe_spatial") != "pe_spatial":
    raise ValueError("이 전처리 경로의 REPA encoder는 pe_spatial입니다.")

datasets = cfg.get("datasets", [])

if len(datasets) != 1 or len(datasets[0].get("subsets", [])) != 1:
    raise ValueError("데이터 원본 하나에 대응하는 dataset/subset 하나를 지정하세요.")

subset = datasets[0]["subsets"][0]


def repo_path(value):
    p = Path(str(value)).expanduser()
    return str((p if p.is_absolute() else REPO / p).resolve())


for key in (
    "pretrained_model_name_or_path",
    "qwen3",
    "vae",
    "output_dir",
    "logging_dir",
):
    if not cfg.get(key):
        raise ValueError(f"필수 설정 누락: {key}")
    cfg[key] = repo_path(cfg[key])

if cfg.get("vocab_pack"):
    cfg["vocab_pack"] = repo_path(cfg["vocab_pack"])

for key in ("image_dir", "cache_dir"):
    if not subset.get(key):
        raise ValueError(f"dataset subset에 {key}가 필요합니다.")
    subset[key] = repo_path(str(subset[key]).format_map(cfg))

if any(
    Path(subset[k]) == SOURCE
    or Path(subset[k]).is_relative_to(SOURCE)
    or SOURCE.is_relative_to(Path(subset[k]))
    for k in ("image_dir", "cache_dir")
):
    raise ValueError("전처리/캐시 폴더는 원본 폴더와 분리하세요.")

if (
    cfg.get("optimizer_type", "").lower() == "adamw"
    and cfg.get("learning_rate", 0) >= 0.01
):
    raise ValueError(
        "AdamW 학습률이 큽니다. Prodigy의 LR=1을 그대로 사용했는지 확인하세요."
    )

if cfg.get("max_train_epochs") and cfg.get("max_train_steps"):
    raise ValueError("max_train_epochs와 max_train_steps 중 하나만 지정하세요.")

if (
    datasets[0].get("batch_size", 1) < 1
    or cfg.get("gradient_accumulation_steps", 1) < 1
):
    raise ValueError("batch와 accumulation은 1 이상이어야 합니다.")

cfg.setdefault("log_with", "tensorboard")
cfg.setdefault("target_res", [RESOLUTION])
cfg.setdefault("caption_shuffle_variants", SHUFFLE_VARIANTS)

if cfg.get("resume"):
    cfg["resume"] = repo_path(cfg["resume"])

if cfg.get("log_with") != "tensorboard":
    raise ValueError(
        "TensorBoard 모니터링을 사용하려면 기존 TOML의 log_with를 tensorboard로 지정하세요."
    )

if not re.fullmatch(r"[A-Za-z0-9_-]+", str(cfg.get("output_name", "anima"))):
    raise ValueError("output_name: 영문·숫자·밑줄·하이픈을 사용하세요.")

# TOML 업로드/복원도 정기 저장과 로그는 런타임 내부에만 씁니다.
for key, subfolder in (("output_dir", "ckpt"), ("logging_dir", "logs")):
    target = Path(cfg[key]).resolve()
    if not target.is_relative_to(Path("/content")) or target.is_relative_to(
        Path("/content/drive")
    ):
        cfg[key] = str(ROOT / "output" / cfg.get("output_name", "anima") / subfolder)
        print(f"{key}: 런타임 내부 경로로 변경했습니다: {cfg[key]}")

CONFIG = ROOT / "configs" / f"{cfg.get('output_name', 'anima')}.toml"
write_toml(CONFIG, cfg)
# TOML에서 생략되는 None 등을 저장된 표현에 맞춥니다.
cfg = read_toml(CONFIG)
SMOKE_OK = None
print(CONFIG.read_text(encoding="utf-8"))
print("저장:", CONFIG)
print(
    "적용된 batch:",
    datasets[0].get("batch_size", 1),
    "| 누적:",
    cfg.get("gradient_accumulation_steps", 1),
    "| 해상도 tier:",
    cfg.get("target_res"),
)
print(
    "학습 목표:",
    f"{cfg['max_train_epochs']} epochs"
    if cfg.get("max_train_epochs")
    else f"{cfg.get('max_train_steps')} steps",
)

if str(cfg["output_dir"]).startswith("/content/") and not str(
    cfg["output_dir"]
).startswith("/content/drive/"):
    print(
        "현재 출력은 임시 디스크입니다. 런타임 삭제 전 수동 저장 후 정지의 GD 보관 옵션이나 백업 다운로드를 사용하세요."
    )

run([PY, "train.py", "--config_file", CONFIG, "--print-config"], cwd=REPO)

In [ ]:
#@title 4-1. 상태 저장과 재개 선택 (TOML 셀 다음에 실행)
ENABLE_STATE_SAVE = True  #@param {type:"boolean"}
STATE_SAVE_INTERVAL = 200  #@param {type:"integer"}
RESUME_MODE = "auto"  #@param ["auto", "fresh", "folder"]
RESUME_STATE_PATH = ""  #@param {type:"string"}
TOTAL_TARGET_STEPS = 0  #@param {type:"integer"}

# 0: 저장된 학습 목표 유지. 예: 1370/2400에서 재개하면 목표는 2400입니다.
if STATE_SAVE_INTERVAL < 1:
    raise ValueError("저장 간격은 1 이상이어야 합니다.")

cfg.update(
    save_state=ENABLE_STATE_SAVE,
    save_state_on_train_end=ENABLE_STATE_SAVE,
    save_every_n_steps=STATE_SAVE_INTERVAL,
)
cfg.pop("checkpointing_epochs", None)
candidate = None

if RESUME_MODE == "folder":
    if not RESUME_STATE_PATH.strip():
        raise ValueError("상태 폴더 경로를 입력하세요.")
    candidate = Path(RESUME_STATE_PATH).expanduser().resolve()
elif RESUME_MODE == "auto":
    candidate = RESTORED_STATE or (Path(cfg["resume"]) if cfg.get("resume") else None)

if candidate:
    state_info = colab_runtime.inspect_state(candidate)
    metadata_path = candidate / "colab_state.json"
    if metadata_path.exists():
        saved_meta = json.loads(metadata_path.read_text())
        if saved_meta["revision"] != REPO_REVISION:
            raise ValueError("상태와 설치 소스 revision이 다릅니다.")
        saved_cfg = saved_meta["config"]
        keys = (
            "network_module",
            "network_dim",
            "network_alpha",
            "network_args",
            "down_init",
            "use_timestep_mask",
            "min_rank",
            "alpha_rank_scale",
            "use_repa",
            "repa_mode",
            "repa_weight",
            "repa_layer",
            "optimizer_type",
            "optimizer_args",
            "learning_rate",
            "lr_scheduler",
            "lr_warmup_steps",
            "gradient_accumulation_steps",
            "mixed_precision",
            "seed",
            "caption_dropout_rate",
            "use_shuffled_caption_variants",
            "caption_shuffle_variants",
            "target_res",
            "svd_slice",
            "channel_scale_alpha",
            "repa_encoder",
            "repa_anneal_steps",
        )
        changed = [k for k in keys if cfg.get(k) != saved_cfg.get(k)]

        # 경로를 제외한 batch·반복·분할도 재개 시 유지합니다.
        def dataset_settings(data):
            result = deepcopy(data)
            for ds in result:
                for sub in ds.get("subsets", []):
                    for key in ("image_dir", "cache_dir"):
                        sub.pop(key, None)
            return result

        if dataset_settings(cfg.get("datasets", [])) != dataset_settings(
            saved_cfg.get("datasets", [])
        ):
            changed.append("datasets(batch/repeats/split)")
        if changed:
            raise ValueError(
                "저장 당시 설정을 불러오세요. 불일치: " + ", ".join(changed)
            )
        RESTORED_MANIFEST = saved_meta
    else:
        print("설정 기록이 없어 자동 비교할 수 없습니다. 저장 당시 TOML을 load하세요.")
    cfg["resume"] = str(candidate)
    cfg["skip_until_initial_step"] = True
    cfg.pop("initial_step", None)
    cfg.pop("initial_epoch", None)
    print("재개 step:", state_info["current_step"], "| 상태:", candidate)
else:
    cfg.pop("resume", None)
    cfg.pop("initial_step", None)
    cfg.pop("initial_epoch", None)
    cfg["skip_until_initial_step"] = False
    print("새 학습")

if TOTAL_TARGET_STEPS < 0:
    raise ValueError("TOTAL_TARGET_STEPS는 0 이상이어야 합니다.")

if TOTAL_TARGET_STEPS:
    cfg.pop("max_train_epochs", None)
    cfg["max_train_steps"] = TOTAL_TARGET_STEPS
    print(
        "총 목표 step 변경: 기존 학습률 스케줄과 완전히 동일한 연장을 보장하지 않습니다."
    )

if candidate and cfg.get("max_train_steps") and not cfg.get("max_train_epochs"):
    if cfg["max_train_steps"] <= state_info["current_step"]:
        raise ValueError("총 목표 step은 저장 step보다 커야 합니다.")

write_toml(CONFIG, cfg)
# TOML에서 생략되는 None 등을 저장된 표현에 맞춥니다.
cfg = read_toml(CONFIG)
SMOKE_OK = None
print(
    "상태 저장:", ENABLE_STATE_SAVE, "| 간격:", STATE_SAVE_INTERVAL, "optimizer steps"
)
print("출력:", cfg["output_dir"], "| TensorBoard:", cfg["logging_dir"])

In [ ]:
#@title 5. 모델 다운로드 — HF 토큰은 Colab Secret의 HF_TOKEN에서 읽음
DOWNLOAD_MODELS = True  #@param {type:"boolean"}

try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if token:
    ENV["HF_TOKEN"] = token

del token

if DOWNLOAD_MODELS:
    run([PY, "tasks.py", "download-model", "anima"], cwd=REPO)
    if cfg.get("vocab_pack"):
        run([PY, "tasks.py", "download-model", "cjk"], cwd=REPO)

for key in ("pretrained_model_name_or_path", "qwen3", "vae"):
    if not Path(cfg[key]).is_file():
        raise FileNotFoundError(f"{key}: {cfg[key]}")

print("모델 경로 확인 완료. REPA 특징 모델은 특징 캐시 단계에서 로드됩니다.")

## 6. 전처리

이미지 리사이즈와 VAE·텍스트 캐시를 생성합니다. REPA를 켜면 PE-Spatial 캐시도 만듭니다.
데이터·모델·해상도·셔플 설정을 바꾸면 새 캐시 경로를 사용하세요.


In [ ]:
#@title 리사이즈·VAE·텍스트·REPA 캐시
CACHE_BATCH_SIZE = 1  #@param {type:"integer"}
VAE_FP32 = True  #@param {type:"boolean"}
VAE_CHUNK_SIZE = 64  #@param {type:"integer"}


def fingerprint():
    # 원본 데이터와 캐시 목록의 변경도 smoke 통과 기록에 반영합니다.
    records = [
        CONFIG.read_text(encoding="utf-8"),
        REPO_REVISION,
        RUNTIME.read_text(encoding="utf-8"),
    ]
    if cfg.get("resume"):
        for p in sorted(Path(cfg["resume"]).rglob("*")):
            if p.is_file():
                st = p.stat()
                records.append((str(p), st.st_size, st.st_mtime_ns))
    for key in ("pretrained_model_name_or_path", "qwen3", "vae"):
        p = Path(cfg[key])
        s = p.stat()
        records.append((str(p), s.st_size, s.st_mtime_ns))
    for folder in (SOURCE, Path(subset["image_dir"]), Path(subset["cache_dir"])):
        if folder.exists():
            for p in sorted(folder.rglob("*")):
                if p.is_file():
                    s = p.stat()
                    records.append((str(p), s.st_size, s.st_mtime_ns))
    return records


def preparation_signature():
    # 파일 내용은 읽지 않고 파일 정보만 비교합니다.
    source_files = []
    for p in images:
        for file in (p, p.with_suffix(".txt")):
            info = file.stat()
            source_files.append([str(file.relative_to(SOURCE)), info.st_size])
    source_files.sort()
    keys = (
        "pretrained_model_name_or_path",
        "qwen3",
        "vae",
        "vocab_pack",
        "target_res",
        "caption_shuffle_variants",
        "caption_tag_dropout_rate",
        "caption_tag_randomize_rate",
    )
    return dict(
        source_files=source_files,
        source_mtimes={
            str(file.relative_to(SOURCE)): file.stat().st_mtime_ns
            for p in images
            for file in (p, p.with_suffix(".txt"))
        },
        repo=REPO_REVISION,
        config={k: cfg.get(k) for k in keys},
        vae_fp32=VAE_FP32,
        vae_chunk=VAE_CHUNK_SIZE,
        model_stats={
            k: [Path(cfg[k]).stat().st_size, Path(cfg[k]).stat().st_mtime_ns]
            for k in keys[:3]
        },
    )


image_dir, cache_dir = Path(subset["image_dir"]), Path(subset["cache_dir"])
manifest = cache_dir / "colab_preparation.json"
signature = preparation_signature()

if RESTORED_MANIFEST and RESTORED_MANIFEST.get("preparation"):
    previous_source = RESTORED_MANIFEST["preparation"].get("source_files")
    if previous_source and previous_source != signature["source_files"]:
        raise RuntimeError("재개 데이터의 파일명·크기가 저장 당시와 다릅니다.")

if manifest.exists():
    if json.loads(manifest.read_text()) != signature:
        raise RuntimeError(
            "전처리 조건이 변경됐습니다. 새로운 image_dir/cache_dir로 실행하세요."
        )
elif any(
    folder.exists() and any(folder.iterdir()) for folder in (image_dir, cache_dir)
):
    raise RuntimeError(
        "출처를 확인할 수 없는 전처리 파일이 있습니다. 비어 있는 별도 경로를 지정하세요."
    )

cache_dir.mkdir(parents=True, exist_ok=True)
# 중단 후 재시도할 때 동일 조건임을 확인할 수 있도록 캐시 생성 전에 기록합니다.
manifest.write_text(
    json.dumps(signature, indent=2, ensure_ascii=False), encoding="utf-8"
)
prep = dict(
    source_image_dir=str(SOURCE),
    resized_image_dir=str(image_dir),
    lora_cache_dir=str(cache_dir),
    target_res=cfg["target_res"],
)
prep_path = ROOT / "configs" / "preprocess_paths.toml"
write_toml(prep_path, prep)
ENV["CONFIG_FILE"] = str(prep_path)
run(
    [PY, "tasks.py", "preprocess-resize", "--target_res", *map(str, cfg["target_res"])],
    cwd=REPO,
)
ENV.pop("CONFIG_FILE", None)
common = ["--cache_dir", cache_dir, "--batch_size", CACHE_BATCH_SIZE, "--recursive"]
run(
    [
        PY,
        "scripts/preprocess/cache_latents.py",
        "--dir",
        image_dir,
        "--vae",
        cfg["vae"],
        "--chunk_size",
        VAE_CHUNK_SIZE,
        *common,
        *(["--no_half_vae"] if VAE_FP32 else []),
    ],
    cwd=REPO,
)
run(
    [
        PY,
        "scripts/preprocess/cache_text_embeddings.py",
        "--dir",
        SOURCE,
        "--match_images_from",
        image_dir,
        "--qwen3",
        cfg["qwen3"],
        "--dit",
        cfg["pretrained_model_name_or_path"],
        "--vocab_pack",
        cfg.get("vocab_pack", ""),
        "--caption_shuffle_variants",
        cfg["caption_shuffle_variants"],
        "--caption_tag_dropout_rate",
        cfg.get("caption_tag_dropout_rate", 0.0),
        "--caption_tag_randomize_rate",
        cfg.get("caption_tag_randomize_rate", 0.0),
        *common,
    ],
    cwd=REPO,
)

if cfg.get("use_repa"):
    run(
        [
            PY,
            "scripts/preprocess/cache_pe_encoder.py",
            "--dir",
            image_dir,
            "--encoder",
            "pe_spatial",
            "--dtype",
            "float32",
            *common,
        ],
        cwd=REPO,
    )

prepared = image_files(image_dir)

if len(prepared) != len(images):
    raise RuntimeError(
        f"원본 {len(images)}장 / 리사이즈 {len(prepared)}장. 전처리 로그를 확인하세요."
    )

PREPARED_SIGNATURE = signature
SMOKE_OK = None
print("전처리 완료:", len(prepared), "장")

## 7. Smoke test (선택)

현재 batch·해상도·학습 기법으로 **5스텝** 실행하고 loss와 저장된 LoRA의 NaN/Inf를 검사합니다. 모델 로딩·SVD 초기화 시간은 별도입니다.

`SMOKE_DIAGNOSTICS`를 켜면 앞 2스텝은 기본 실행, 나머지는 진단을 포함합니다. 일부 LoRA 모듈의 gradient·업데이트·T-LoRA rank와 REPA 지표를 기록합니다.

- cosine은 연속 batch의 gradient 비교이며, 주 loss와 REPA의 충돌 지표가 아닙니다.
- 시간·메모리는 초기화와 버킷 차이가 섞인 참고값입니다. gradient 누적 중에는 update norm이 0일 수 있습니다.
- 통과는 실행 확인입니다. 장기 학습 품질·SVD 효과·모든 버킷의 메모리 적합성을 보장하지 않습니다.


In [ ]:
#@title 현재 설정으로 짧은 실제 학습
SMOKE_DIAGNOSTICS = False  #@param {type:"boolean"}
DIAGNOSTIC_MODULES = 3  #@param {type:"integer"}
SMOKE_STEPS = 5  #@param {type:"integer"}

if SMOKE_DIAGNOSTICS and SMOKE_STEPS < 5:
    raise ValueError("진단 포함 smoke는 5 steps 이상이어야 합니다.")

if DIAGNOSTIC_MODULES < 1:
    raise ValueError("측정 모듈 수는 1 이상이어야 합니다.")

if SMOKE_STEPS < 2:
    raise ValueError("SMOKE_STEPS는 2 이상으로 설정하세요.")

if read_toml(CONFIG) != cfg:
    raise RuntimeError("TOML이 변경됐습니다. 설정 셀부터 다시 실행하세요.")

if preparation_signature() != PREPARED_SIGNATURE:
    raise RuntimeError("데이터/모델 변경: 전처리 셀부터 다시 실행하세요.")

SMOKE_OK = None
smoke = deepcopy(cfg)

for key in (
    "max_train_epochs",
    "resume",
    "initial_epoch",
    "initial_step",
    "checkpointing_epochs",
    "save_every_n_epochs",
    "skip_until_initial_step",
):
    smoke.pop(key, None)

smoke_dir = ROOT / "smoke" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
smoke_dir.mkdir(parents=True)
progress = smoke_dir / "progress.jsonl"
smoke.update(
    max_train_steps=SMOKE_STEPS,
    output_dir=str(smoke_dir),
    output_name="smoke",
    logging_dir=str(smoke_dir / "logs"),
    log_with="tensorboard",
    log_every_n_steps=1,
    progress_jsonl=str(progress),
    save_every_n_steps=SMOKE_STEPS,
    save_state=False,
    save_state_on_train_end=False,
    save_model_as="safetensors",
)
smoke_path = smoke_dir / "smoke.toml"
write_toml(smoke_path, smoke)
before = fingerprint()
print(f"모델·LoRA 초기화 후 {SMOKE_STEPS} optimizer steps를 학습하고 저장합니다.")
print("loss 검사는 NaN/Inf 여부를 확인하며, 증가 추세는 실패 조건이 아닙니다.")
probe_path = smoke_dir / "diagnostics.jsonl"
run(
    [
        PY,
        RUNTIME,
        "--config",
        smoke_path,
        "--probe",
        probe_path,
        "--module-limit",
        DIAGNOSTIC_MODULES,
        *(["--diagnostics"] if SMOKE_DIAGNOSTICS else []),
    ],
    cwd=REPO,
    log=smoke_dir / "console.log",
)
records = [
    json.loads(line) for line in progress.read_text().splitlines() if line.strip()
]
steps = [r for r in records if r.get("ev") == "step"]
ends = [r for r in records if r.get("ev") == "run_end"]
losses = [v for r in steps for k, v in r.items() if "loss" in k.lower()]

if (
    not ends
    or ends[-1].get("status") != "ok"
    or ends[-1].get("final_step", 0) < SMOKE_STEPS
):
    raise RuntimeError("smoke 완료 step/종료 상태 검증 실패")

if not losses or not all(
    isinstance(v, (int, float)) and math.isfinite(v) for v in losses
):
    raise RuntimeError("loss가 없거나 NaN/Inf입니다.")

py(
    r"""
import sys, torch
from pathlib import Path
from safetensors import safe_open
files = list(Path(sys.argv[1]).glob('smoke*.safetensors'))
if not files: raise RuntimeError('저장된 LoRA가 없습니다.')
count = 0
for path in files:
    with safe_open(path, framework='pt', device='cpu') as f:
        for key in f.keys():
            if not torch.isfinite(f.get_tensor(key)).all(): raise RuntimeError(f'NaN/Inf: {path.name}:{key}')
            count += 1
if not count: raise RuntimeError('빈 체크포인트입니다.')
print('저장 tensor 검사 PASS:', count)
""",
    str(smoke_dir),
)

if fingerprint() != before:
    raise RuntimeError("smoke 중 설정/데이터가 변경됐습니다.")

probe_report = colab_runtime.summarize_probe(probe_path)

if SMOKE_DIAGNOSTICS:
    if "diagnostic" not in probe_report:
        raise RuntimeError("진단 구간이 실행되지 않았습니다.")
    keys = probe_report["metrics"]
    if not any(k.endswith("/grad_norm") for k in keys):
        raise RuntimeError(
            "일반 LoRA 그라디언트 지표가 수집되지 않았습니다. 기법/모듈 호환성을 확인하세요."
        )
    if cfg.get("use_timestep_mask") and not any(
        k.endswith("/active_rank_mean") for k in keys
    ):
        raise RuntimeError("T-LoRA 활성 rank 지표가 없습니다.")
    if cfg.get("use_repa") and "repa/align_loss" not in keys:
        raise RuntimeError("REPA가 켜져 있지만 smoke에서 손실을 수집하지 못했습니다.")
    if (
        cfg.get("use_repa")
        and cfg.get("repa_mode", "relational") == "relational"
        and "repa/heatmap_conc" not in keys
    ):
        raise RuntimeError("REPA heatmap 진단을 수집하지 못했습니다.")
    print("진단 수집 PASS. 기법 효과 판단: 표본 부족")

print(json.dumps(probe_report, ensure_ascii=False, indent=2))
print("시간은 로딩·저장·로그 처리를 제외한 micro-step 참고값입니다.")
SMOKE_OK = before
report = dict(
    diagnostics=probe_report,
    diagnostics_enabled=SMOKE_DIAGNOSTICS,
    status="PASS",
    steps=SMOKE_STEPS,
    fingerprint=before,
    repo=REPO_REVISION,
    loss_min=min(losses),
    loss_max=max(losses),
    time=datetime.now(timezone.utc).isoformat(),
)
(smoke_dir / "result.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("SMOKE PASS:", smoke_dir)

## 8. TensorBoard

기본은 Colab 내부 보기입니다. `PUBLIC_TUNNEL`을 켜면 **URL을 아는 누구나 로그를 볼 수 있는** 임시 공개 주소가 생성됩니다.

복원한 로그는 `TB_LOG_MODE=current`, `TB_CONTINUOUS=True`로 저장 스텝부터 이어 봅니다. `TB_CONTINUOUS=False`는 세션별 로그를 표시합니다.


In [ ]:
#@title TensorBoard 시작 및 선택적 공개 터널
TB_LOG_MODE = "current"  #@param ["current", "folder"]
TB_LOG_FOLDER = ""  #@param {type:"string"}
TB_CONTINUOUS = True  #@param {type:"boolean"}
TB_PORT = 6006  #@param {type:"integer"}
PUBLIC_TUNNEL = False  #@param {type:"boolean"}


def stop_service(name):
    service = PROCESSES.pop(name, None)
    if service:
        proc, handle = service
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
        handle.close()


def start_service(name, command):
    stop_service(name)
    log = ROOT / f"{name}.log"
    handle = log.open("w")
    proc = subprocess.Popen(
        [str(x) for x in command],
        cwd=REPO,
        env=ENV,
        stdout=handle,
        stderr=subprocess.STDOUT,
    )
    PROCESSES[name] = (proc, handle)
    return proc, log


stop_service("cloudflared")

if TB_LOG_MODE == "folder" and not TB_LOG_FOLDER.strip():
    raise ValueError("이전 TensorBoard 로그 폴더를 입력하세요.")

tb_root = (
    Path(TB_LOG_FOLDER).expanduser().resolve()
    if TB_LOG_MODE == "folder"
    else Path(cfg["logging_dir"])
)

if TB_LOG_MODE == "folder" and not tb_root.is_dir():
    raise FileNotFoundError(tb_root)

tb_view = tb_root / "colab_timeline" if TB_CONTINUOUS else tb_root
tb_view.mkdir(parents=True, exist_ok=True)
print("TensorBoard 읽기 경로:", tb_view)

if TB_LOG_MODE == "folder":
    print(
        "이 옵션은 이전 로그 보기용입니다. 새 학습 기록은 TOML의 logging_dir에 저장됩니다."
    )

tb, tb_log = start_service(
    "tensorboard",
    [
        PY,
        "-m",
        "tensorboard.main",
        "--logdir",
        tb_view,
        "--host",
        "127.0.0.1",
        "--port",
        TB_PORT,
    ],
)

for _ in range(60):
    if tb.poll() is not None:
        raise RuntimeError(tb_log.read_text())
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{TB_PORT}/", timeout=2).close()
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("TensorBoard 시작 시간 초과: " + str(tb_log))

from google.colab import output

output.serve_kernel_port_as_iframe(TB_PORT, height=700)

if PUBLIC_TUNNEL:
    binary = TOOLS / "cloudflared"
    if not binary.exists():
        release = fetch_json(
            "https://api.github.com/repos/cloudflare/cloudflared/releases/latest"
        )
        asset = next(
            a for a in release["assets"] if a["name"] == "cloudflared-linux-amd64"
        )
        urllib.request.urlretrieve(asset["browser_download_url"], binary)
        binary.chmod(0o755)
    tunnel, tunnel_log = start_service(
        "cloudflared",
        [binary, "tunnel", "--url", f"http://127.0.0.1:{TB_PORT}", "--no-autoupdate"],
    )
    for _ in range(60):
        if tunnel.poll() is not None:
            raise RuntimeError(tunnel_log.read_text())
        match = re.search(
            r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_log.read_text()
        )
        if match:
            print("공개 TensorBoard:", match.group())
            break
        time.sleep(1)
    else:
        raise RuntimeError("터널 주소 발급 시간 초과: " + str(tunnel_log))

In [ ]:
#@title 9. 본 학습
START_TRAINING = False  #@param {type:"boolean"}
SAVE_STOP_TO_DRIVE = False  #@param {type:"boolean"}
# 체크하면 시작 전에 GD를 마운트합니다. 정기 저장은 여전히 런타임 내부입니다.
# 셀 실행이 끝나도 학습은 계속됩니다. 아래 버튼으로 저장 후 정지하세요.

if START_TRAINING:
    if PROCESSES.get("training") is not None and PROCESSES["training"].poll() is None:
        raise RuntimeError("이미 학습 중입니다. 저장 없이 중단하려면 9-A 셀을 실행하세요.")
    if read_toml(CONFIG) != cfg:
        raise RuntimeError("TOML이 변경됐습니다. 설정 셀부터 다시 실행하세요.")
    output_dir = Path(cfg["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    if any(output_dir.rglob("*.safetensors")) and not cfg.get("resume"):
        raise RuntimeError(
            "기존 체크포인트가 있습니다. 새 출력 경로 또는 명시적인 resume을 사용하세요."
        )
    saved = output_dir / f"{cfg.get('output_name', 'anima')}_colab.toml"
    shutil.copy2(CONFIG, saved)
    log = output_dir / ("train_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".log")
    session_path = output_dir / "colab_session.json"
    session_path.write_text(
        json.dumps(
            dict(
                revision=REPO_REVISION,
                root=str(ROOT),
                repo=str(REPO),
                preparation=PREPARED_SIGNATURE,
            ),
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    control_path = output_dir / (
        "control_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f") + ".json"
    )
    PROCESSES["training"] = colab_runtime.start_training_controls(
        [
            PY,
            RUNTIME,
            "--config",
            saved,
            "--timeline",
            "--session",
            session_path,
            "--control",
            control_path,
        ],
        cwd=REPO,
        env=ENV,
        log=log,
        control=control_path,
        save_to_drive=SAVE_STOP_TO_DRIVE,
    )
    print("학습 로그:", log)
    print(
        "GD 보관 후 새 세션: 3-1에서 state_folder를 선택하고 GD의 manual-.../state를 지정하세요."
    )
    print(
        "같은 세션 재개: 4-1에서 folder와 저장된 state 경로를 선택한 뒤 본 학습을 실행하세요."
    )
else:
    print("START_TRAINING을 켜고 이 셀을 실행하면 본 학습을 시작합니다.")

In [ ]:
#@title 현재 학습 로그 확인 — 실행할 때마다 갱신
LAST_LINES = 80  #@param {type:"integer"}

from pathlib import Path
from datetime import datetime
import re

log_value = globals().get("log")
if log_value is None:
    raise RuntimeError("본 학습을 시작한 뒤 실행하세요.")
log_path = Path(log_value)
if not log_path.is_file():
    raise FileNotFoundError(f"로그 파일이 없습니다: {log_path}")

process = PROCESSES.get("training")
if process is not None:
    code = process.poll()
    print("학습 상태:", "실행 중" if code is None else f"종료 — 코드 {code}")

info = log_path.stat()
print("로그:", log_path)
print("마지막 기록:", datetime.fromtimestamp(info.st_mtime))
print("-" * 60)

with log_path.open("rb") as handle:
    handle.seek(max(0, info.st_size - 256 * 1024))
    text = handle.read().decode("utf-8", errors="replace")

text = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text)
lines = text.replace("\r\n", "\n").replace("\r", "\n").splitlines()
lines = [line for line in lines if line.strip()]
print("\n".join(lines[-max(1, LAST_LINES):]) or "아직 기록된 로그가 없습니다.")


In [ ]:
#@title 9-A. 저장 없이 중단 / 처음부터 학습 준비
STOP_WITHOUT_SAVE = False  #@param {type:"boolean"}
PREPARE_FRESH_RUN = True  #@param {type:"boolean"}
# 실행 중인 학습만 종료합니다. 이미 저장된 파일과 데이터·캐시는 유지합니다.

if STOP_WITHOUT_SAVE:
    import signal

    process = PROCESSES.get("training")
    if process is not None and process.poll() is None:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=10)
    print("학습 중단 완료. 추가 저장은 요청하지 않았습니다.")

    if PREPARE_FRESH_RUN:
        fresh_root = ROOT / "output" / ("fresh_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
        for key in ("resume", "initial_step", "initial_epoch"):
            cfg.pop(key, None)
        cfg["skip_until_initial_step"] = False
        cfg["output_dir"] = str(fresh_root / "ckpt")
        cfg["logging_dir"] = str(fresh_root / "logs")
        RESTORED = RESTORED_STATE = RESTORED_MANIFEST = None
        write_toml(CONFIG, cfg)
        cfg = read_toml(CONFIG)
        SMOKE_OK = None
        print("새 출력:", fresh_root)
        print("데이터·캐시는 그대로 사용합니다. 8번 TensorBoard → 9번 본 학습을 실행하세요. Smoke는 필요할 때 별도로 실행합니다.")
else:
    print("중단하려면 STOP_WITHOUT_SAVE를 체크하고 실행하세요.")


In [ ]:
#@title 9-1. 완료된 상태와 TensorBoard 로그 백업 / 내려받기
CREATE_BACKUP = False  #@param {type:"boolean"}
DOWNLOAD_BACKUP = False  #@param {type:"boolean"}
BACKUP_STATE_PATH = ""  #@param {type:"string"}
BACKUP_FOLDER = ""  #@param {type:"string"}
# 학습이 끝났거나 중지된 뒤 실행하세요. 빈 상태 경로는 가장 큰 저장 step을 선택합니다.
# 현재 메모리의 미저장 학습을 새로 저장하는 기능이 아닙니다.
if (
    (CREATE_BACKUP or DOWNLOAD_BACKUP)
    and PROCESSES.get("training") is not None
    and PROCESSES["training"].poll() is None
):
    raise RuntimeError("학습을 저장 후 정지한 뒤 백업 셀을 실행하세요.")

states = colab_runtime.list_states(cfg["output_dir"], complete_only=True)

for step, folder in states:
    print("저장 step:", step, "|", folder)

if CREATE_BACKUP:
    if not states and not BACKUP_STATE_PATH.strip():
        raise ValueError("완료된 상태 저장이 없습니다.")
    state_folder = (
        Path(BACKUP_STATE_PATH).expanduser().resolve()
        if BACKUP_STATE_PATH.strip()
        else states[-1][1]
    )
    meta_file = state_folder / "colab_state.json"
    if not meta_file.is_file():
        raise ValueError(
            "백업할 상태의 설정 기록이 없습니다. 원본 상태 폴더와 저장 당시 TOML을 직접 보관하세요."
        )
    meta = json.loads(meta_file.read_text())
    if meta["revision"] != REPO_REVISION:
        raise ValueError("상태의 소스 revision이 다릅니다.")
    backup_cfg = meta["config"]
    backup_root = (
        Path(BACKUP_FOLDER).expanduser()
        if BACKUP_FOLDER.strip()
        else Path(cfg["output_dir"]).parent / "backups"
    )
    backup_root.mkdir(parents=True, exist_ok=True)
    step = colab_runtime.inspect_state(state_folder)["current_step"]
    backup = (
        backup_root
        / f"{cfg['output_name']}_step{step}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.zip"
    )
    backup_toml = ROOT / "configs" / "backup_snapshot.toml"
    write_toml(backup_toml, backup_cfg)
    colab_runtime.create_bundle(
        backup,
        state_folder,
        backup_cfg,
        backup_toml,
        REPO_REVISION,
        meta["root"],
        meta["repo"],
        meta.get("preparation"),
    )
    print("백업 완료:", backup, "| 크기 GiB:", round(backup.stat().st_size / 2**30, 3))
    print(
        "포함: 학습 상태·TOML·TensorBoard 로그. 원본 데이터·모델·전처리 캐시는 별도 보관하세요."
    )
    if DOWNLOAD_BACKUP:
        from google.colab import files

        files.download(str(backup))
elif DOWNLOAD_BACKUP:
    print("CREATE_BACKUP도 켜면 백업을 만들고 내려받습니다.")

In [ ]:
#@title 10. TOML 내려받기 / 모니터링 종료
DOWNLOAD_TOML = False  #@param {type:"boolean"}
STOP_MONITORING = False  #@param {type:"boolean"}

if DOWNLOAD_TOML:
    from google.colab import files

    files.download(str(CONFIG))

if STOP_MONITORING:
    stop_service("cloudflared")
    stop_service("tensorboard")
    print("모니터링 종료")

print("TOML:", CONFIG)
print("체크포인트:", cfg["output_dir"])

## 문제 해결 / 재개

- **OOM:** batch·해상도·swap·REPA를 조정하고 전처리·smoke를 다시 실행하세요.
- **NaN/Inf:** 정밀도·학습률을 확인하세요. Prodigy의 LR=1을 AdamW에 사용하면 안 됩니다.
- **새 런타임에서 재개:** 설치·데이터 준비 → 3-1 복원 → TOML·4-1 재개 확인 → 모델·전처리 → smoke(선택) → TensorBoard → 본 학습. GPU에서 optimizer 상태를 불러오는 것은 본 학습 시작 시입니다.
- **GD 수동 저장 복원:** 3-1의 `state_folder`에 GD의 `manual-.../state` 경로를 지정하세요. 같은 폴더의 `lora.safetensors`는 이미지 생성에 사용합니다.
- **런타임 삭제:** 내부 파일은 복구할 수 없습니다. GD 보관이나 다운로드 완료를 확인하세요.
